In [ ]:
import geopandas as gpd
import pandas as pd

import sys
sys.path.append("../")
import src.paths as PATHS

import numpy as np
import math
from shapely.geometry import Point

In [ ]:
test_erosion_data = gpd.read_file(PATHS.TEST_DIR / "assets" / "river_bank_points_for_tests.geojson")

test_erosion_data.info()

In [ ]:
test_erosion_data["ahn_version"].value_counts()

In [ ]:
test_erosion_data["status"].value_counts() / len(test_erosion_data)

In [ ]:
def randomly_shift_point(point: Point, distance: float, custom_z=None) -> Point:
    """
    Shift a Shapely Point in a random direction by a set distance.

    :param point: The original Shapely Point.
    :param distance: The distance to shift the point.
    :return: A new Shapely Point shifted by the given distance.

    TODO: shift also in Z direction if need be; right now we just keep the Z
    """
    # Generate a random angle in radians
    angle = np.random.uniform(0, 2 * math.pi)
    
    # Calculate the new coordinates
    new_x = point.x + distance * math.cos(angle)
    new_y = point.y + distance * math.sin(angle)
    try:
        new_z = point.z
    except:
        new_z = custom_z
    
    return Point(new_x, new_y, new_z)

In [ ]:
SHIFT_DISTANCE = 1  # metre

# some points don't have a Z coordinate, replace their Z with the mean of all
mean_z = []
no_2d_points = 0
for pt in test_erosion_data["geometry"]:
    try:
        mean_z.append(pt.z)
    except:
        no_2d_points += 1

print(f"There is a total of {no_2d_points} 2D points with no Z coordinate.")
mean_z = np.mean(mean_z)

In [ ]:
fake_data = []

for _,row in test_erosion_data[test_erosion_data["ahn_version"] == 5].iterrows():
    old_row = row.to_dict().copy()
    for fake_ahn_version in [6, 7, 8, 9]:
        # pretend that future ahns exist
        new_row = {
            "status": np.random.choice(test_erosion_data["status"].unique(), p = test_erosion_data["status"].value_counts() / len(test_erosion_data)),  # randomly assign status like in the real data
            "location_id": old_row["location_id"],  # TODO: perhaps replace to make sure that we stay inside the rectangle?
            "ahn_version": fake_ahn_version,
            "geometry": randomly_shift_point(old_row["geometry"], distance=SHIFT_DISTANCE, custom_z=mean_z)
        }
        fake_data.append(new_row)
        
        old_row = new_row.copy()

fake_data = gpd.GeoDataFrame(fake_data, crs=test_erosion_data.crs)

test_erosion_data = pd.concat([test_erosion_data, fake_data])

In [ ]:
test_erosion_data.info()

In [ ]:
test_erosion_data.to_file(PATHS.TEST_DIR / "assets" / "river_bank_points_for_tests.geojson", driver="GeoJSON")

In [ ]:
test_erosion_data.iloc[745:755]